# PyTorch Fundamentals for LLM Development

This notebook provides a comprehensive introduction to PyTorch, specifically designed for building Large Language Models. We'll cover everything from basic tensor operations to neural network training, with a focus on concepts you'll need for transformer architectures.

## Table of Contents
1. [Introduction to PyTorch](#1-introduction-to-pytorch)
2. [Tensors: The Foundation](#2-tensors-the-foundation)
3. [Tensor Operations](#3-tensor-operations)
4. [Automatic Differentiation](#4-automatic-differentiation)
5. [Building Neural Networks](#5-building-neural-networks)
6. [Training Neural Networks](#6-training-neural-networks)
7. [Working with GPUs](#7-working-with-gpus)
8. [Practice Exercises](#8-practice-exercises)



## 1. Introduction to PyTorch

### What is PyTorch?

PyTorch is a deep learning framework that provides:
- **Tensors**: Multi-dimensional arrays (like NumPy) with GPU support
- **Automatic Differentiation**: Computes gradients automatically for training
- **Neural Network Modules**: Pre-built components for building models
- **Optimization Tools**: Algorithms for training neural networks

### Why PyTorch for LLMs?

- **Dynamic Computation Graphs**: Flexibility for variable-length sequences (critical for text)
- **Pythonic**: Easy to debug and understand
- **Ecosystem**: Libraries like Hugging Face Transformers are built on PyTorch
- **Research-Friendly**: Most LLM research papers use PyTorch

Let's start by importing PyTorch and checking our setup:

In [ ]:
import sys
print(sys.executable)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Check PyTorch version
print(f"PyTorch version: {torch.__version__}")

# Check if CUDA (NVIDIA GPU support) is available
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")



## 2. Tensors: The Foundation

### What are Tensors?

Tensors are the fundamental data structure in PyTorch. Think of them as multi-dimensional arrays:

- **0D Tensor (Scalar)**: A single number → `5`
- **1D Tensor (Vector)**: A sequence → `[1, 2, 3]`
- **2D Tensor (Matrix)**: A table → `[[1, 2], [3, 4]]`
- **3D+ Tensor**: Higher dimensional data

### Why Tensors Matter for LLMs

In LLMs, text is represented as tensors:
- Each word/token → vector (embeddings)
- A sentence → 2D tensor (sequence of vectors)
- A batch of sentences → 3D tensor

Example: "Hello world" might be represented as:
```
[
  [0.2, 0.5, 0.1, ...],  # "Hello" embedding (768 dimensions)
  [0.8, 0.1, 0.3, ...]   # "world" embedding (768 dimensions)
]
```

### Creating Tensors

In [ ]:
# 0D Tensor (Scalar)
scalar = torch.tensor(42)
print(f"Scalar: {scalar}")
print(f"Shape: {scalar.shape}")
print(f"Number of dimensions: {scalar.ndim}\n")

# 1D Tensor (Vector)
vector = torch.tensor([1, 2, 3, 4, 5])
print(f"Vector: {vector}")
print(f"Shape: {vector.shape}")
print(f"Number of dimensions: {vector.ndim}\n")

# 2D Tensor (Matrix)
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])
print(f"Matrix:\n{matrix}")
print(f"Shape: {matrix.shape}  # (rows, columns)")
print(f"Number of dimensions: {matrix.ndim}\n")

# 3D Tensor (Common in LLMs: batch of sequences of embeddings)
tensor_3d = torch.tensor([
    [[1, 2], [3, 4]],  # First sequence
    [[5, 6], [7, 8]]   # Second sequence
])
print(f"3D Tensor:\n{tensor_3d}")
print(f"Shape: {tensor_3d.shape}  # (batch_size, sequence_length, embedding_dim)")
print(f"Number of dimensions: {tensor_3d.ndim}")

### Data Types (dtype)

Tensors have data types that determine:
- Memory usage
- Computation precision
- Compatible operations

Common types for LLMs:
- `torch.float32` (default): Standard precision for training
- `torch.float16` / `torch.bfloat16`: Half precision for memory efficiency
- `torch.int64`: For token indices
- `torch.bool`: For attention masks

In [ ]:
# Integer tensor (default: int64)
int_tensor = torch.tensor([1, 2, 3])
print(f"Integer tensor dtype: {int_tensor.dtype}\n")

# Float tensor (default: float32)
float_tensor = torch.tensor([1.0, 2.0, 3.0])
print(f"Float tensor dtype: {float_tensor.dtype}\n")

# Converting between types
converted = int_tensor.to(torch.float32)
print(f"Converted to float32: {converted}")
print(f"New dtype: {converted.dtype}\n")

# Creating tensors with specific dtype
specific_dtype = torch.tensor([1, 2, 3], dtype=torch.float16)
print(f"Specific dtype: {specific_dtype.dtype}")

### Common Tensor Creation Functions

Instead of manually creating tensors, PyTorch provides utility functions:

In [ ]:
# Zeros
zeros = torch.zeros(3, 4)
print(f"Zeros tensor:\n{zeros}\n")

# Ones
ones = torch.ones(2, 3)
print(f"Ones tensor:\n{ones}\n")

# Random values (uniform distribution [0, 1))
random = torch.rand(3, 3)
print(f"Random tensor:\n{random}\n")

# Random values (normal distribution, mean=0, std=1)
randn = torch.randn(3, 3)
print(f"Random normal tensor:\n{randn}\n")

# Range of values (like Python's range)
arange = torch.arange(0, 10, 2)  # start, end, step
print(f"Arange tensor: {arange}\n")

# Identity matrix (important for attention mechanisms)
identity = torch.eye(4)
print(f"Identity matrix:\n{identity}")

### Random Seeds for Reproducibility

**Critical for Research & Debugging**: Setting seeds ensures random operations are reproducible.

In [ ]:
# Without seed - different results each time
print("Without seed:")
print(torch.randn(3))
print(torch.randn(3))

# With seed - same results
print("\nWith seed:")
torch.manual_seed(42)
print(torch.randn(3))
torch.manual_seed(42)
print(torch.randn(3))

### Converting Between NumPy and PyTorch

You'll often need to convert between NumPy arrays and PyTorch tensors:

In [ ]:
# NumPy to PyTorch
numpy_array = np.array([[1, 2], [3, 4]])
print(f"NumPy array:\n{numpy_array}\n")

# Method 1: torch.from_numpy() - shares memory!
tensor_shared = torch.from_numpy(numpy_array)
print(f"Tensor (shared memory):\n{tensor_shared}\n")

# Modifying numpy array affects tensor
numpy_array[0, 0] = 999
print(f"After modifying NumPy array:")
print(f"NumPy: {numpy_array[0, 0]}")
print(f"Tensor: {tensor_shared[0, 0]}\n")

In [ ]:

# Method 2: torch.tensor() - creates a copy
numpy_array = np.array([[1, 2], [3, 4]])
tensor_copy = torch.tensor(numpy_array)
numpy_array[0, 0] = 999
print(f"With copy:")
print(f"NumPy: {numpy_array[0, 0]}")
print(f"Tensor: {tensor_copy[0, 0]}\n")

# PyTorch to NumPy
tensor = torch.tensor([[5, 6], [7, 8]])
numpy_from_tensor = tensor.numpy()
print(f"Converted back to NumPy:\n{numpy_from_tensor}")



## 3. Tensor Operations

### Basic Arithmetic

Tensor operations are element-wise by default:

In [ ]:
# Create two tensors
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

# Element-wise operations
print(f"Addition: {a + b}")
print(f"Subtraction: {a - b}")
print(f"Multiplication: {a * b}")
print(f"Division: {a / b}")
print(f"Power: {a ** 2}")

# Operations with scalars (broadcasting)
print(f"\nScalar addition: {a + 10}")
print(f"Scalar multiplication: {a * 2}")

### Shape Manipulation

**Critical for LLMs**: You'll frequently need to reshape tensors to match expected dimensions.

In [ ]:
# Create a tensor
x = torch.arange(12)
print(f"Original: {x}")
print(f"Shape: {x.shape}\n")

# Reshape to 2D (3 rows, 4 columns)
x_2d = x.reshape(3, 4)
print(f"Reshaped to 3x4:\n{x_2d}")
print(f"Shape: {x_2d.shape}\n")

# Reshape to 3D (2 batches, 2 sequences, 3 features)
x_3d = x.reshape(2, 2, 3)
print(f"Reshaped to 2x2x3:\n{x_3d}")
print(f"Shape: {x_3d.shape}\n")

# Use -1 to infer dimension
x_auto = x.reshape(3, -1)  # 3 rows, auto-calculate columns
print(f"Auto-calculated shape:\n{x_auto}")
print(f"Shape: {x_auto.shape}\n")

# View (similar to reshape, but requires contiguous memory)
x_view = x.view(4, 3)
print(f"View:\n{x_view}")
print(f"Shape: {x_view.shape}")

### Transpose and Permute

Used extensively in attention mechanisms:

In [ ]:
# Create a 2D tensor
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])
print(f"Original matrix ({matrix.shape}):\n{matrix}\n")

# Transpose (flip dimensions)
transposed = matrix.T
print(f"Transposed ({transposed.shape}):\n{transposed}\n")

### permute
permute is a PyTorch method used to rearrange the order of dimensions (axes) in a tensor.
While transpose typically swaps just two dimensions (like flipping a row and a column), permute allows you to shuffle all dimensions into any order you specify.

In [ ]:
# For higher dimensions, use permute
tensor_3d = torch.randn(2, 3, 4)  # (batch, seq_len, features)
print(f"Original 3D shape: {tensor_3d.shape}")

# Rearrange dimensions: (batch, seq_len, features) -> (batch, features, seq_len)
permuted = tensor_3d.permute(0, 2, 1)
print(f"Permuted shape: {permuted.shape}")

### Indexing and Slicing

Similar to NumPy, but critical for selecting tokens, batches, etc.:

In [ ]:
# Create a tensor
tensor = torch.arange(20).reshape(4, 5)
print(f"Tensor:\n{tensor}\n")

# Index a single element
print(f"Element at [1, 2]: {tensor[1, 2]}\n")

# Slice a row
print(f"First row: {tensor[0]}")
print(f"Last row: {tensor[-1]}\n")

# Slice a column
print(f"Second column: {tensor[:, 1]}\n")

# Slice a sub-tensor
print(f"Sub-tensor [1:3, 2:4]:\n{tensor[1:3, 2:4]}\n")

# Boolean indexing (masking)
mask = tensor > 10
print(f"Elements > 10: {tensor[mask]}")

This is **Slicing**, a way to extract a specific "window" or sub-section of a tensor.

In PyTorch and NumPy, slicing follows the syntax: `[row_range, column_range]`. The range is defined as `start:stop`, where the **start is inclusive** and the **stop is exclusive** (meaning it stops just before that number).

**Breaking Down the Slice: `[1:3, 2:4]`**

To find the result, we look at your original tensor and treat it like a grid with coordinates:

- **1. The Rows: `1:3`**

* This tells the computer: "Start at index **1** and stop before index **3**."
* This selects **Row 1** and **Row 2**.
* *Row 0 and Row 3 are ignored.*

- **2. The Columns: `2:4`**

* This tells the computer: "Start at index **2** and stop before index **4**."
* This selects **Column 2** and **Column 3**.
* *Columns 0, 1, and 4 are ignored.*

**The Intersection**

When you overlap these two selections, you get the following values:

|  | Col 2 | Col 3 |
|  |  |  |
| **Row 1** | 7 | 8 |
| **Row 2** | 12 | 13 |

**Key Rules of Slicing**

* **Inclusive:Exclusive** — `1:3` always means "Give me 1 and 2, but not 3."
* **Counting starts at 0** — The first row is index 0.
* **The Shape** — You can calculate the shape of the result by subtracting the start from the stop ( rows, and  columns), giving you a  result.

### Matrix Multiplication

**THE most important operation in neural networks!**

Matrix multiplication is how neural networks transform data. In LLMs:
- Embeddings are multiplied by weight matrices
- Attention uses matrix multiplication for queries, keys, values
- Feed-forward layers are matrix multiplications

In [ ]:
# Create two matrices
A = torch.tensor([[1, 2],
                  [3, 4],
                  [5, 6]])  # Shape: (3, 2)

B = torch.tensor([[7, 8, 9],
                  [10, 11, 12]])  # Shape: (2, 3)

print(f"Matrix A ({A.shape}):\n{A}\n")
print(f"Matrix B ({B.shape}):\n{B}\n")

# Matrix multiplication: (3, 2) @ (2, 3) = (3, 3)
C = torch.matmul(A, B)
print(f"A @ B ({C.shape}):\n{C}\n")

# Alternative syntax
C_alt = A @ B
print(f"Same result using @:\n{C_alt}\n")

# IMPORTANT: Dimensions must be compatible!
# (m, n) @ (n, p) = (m, p)
# The inner dimensions (n) must match!

# Example with batches (common in LLMs)
batch_A = torch.randn(32, 10, 512)  # 32 batches, 10 tokens, 512 features
batch_B = torch.randn(32, 512, 256) # 32 batches, 512 features, 256 output

batch_C = batch_A @ batch_B
print(f"Batched matmul: {batch_A.shape} @ {batch_B.shape} = {batch_C.shape}")

### Aggregation Operations

Used for statistics, pooling, and attention:

In [ ]:
# Create a tensor
tensor = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]])
print(f"Tensor:\n{tensor}\n")

# Sum
print(f"Sum of all elements: {tensor.sum()}")
print(f"Sum along rows (dim=0): {tensor.sum(dim=0)}")
print(f"Sum along columns (dim=1): {tensor.sum(dim=1)}\n")

# Mean
print(f"Mean: {tensor.mean()}")
print(f"Mean along rows: {tensor.mean(dim=0)}")
print(f"Mean along columns: {tensor.mean(dim=1)}\n")

# Max and Min
print(f"Max: {tensor.max()}")
print(f"Min: {tensor.min()}")

# Argmax (index of maximum value)
print(f"Argmax: {tensor.argmax()}")
print(f"Argmax along columns: {tensor.argmax(dim=1)}")

### Broadcasting

PyTorch automatically expands tensors to match shapes when possible:

In [ ]:
#  SCALAR AND TENSOR 
# Scalar (a single number) is treated like a tensor of the same shape as 'tensor'.
# PyTorch "stretches" the 10 into [[10, 10, 10], [10, 10, 10]] so it can add element-by-element.
tensor = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])
scalar = 10
print(f"Tensor + Scalar:\n{tensor + scalar}\n")

#  1D AND 2D TENSORS 
# The vector [10, 20, 30] has shape (3,). 
# PyTorch prepends a dimension to make it (1, 3), then "clones" it downward
# to become a (2, 3) matrix so it matches the original tensor's rows.
vector = torch.tensor([10, 20, 30])
print(f"Tensor + Vector (broadcasts across rows):\n{tensor + vector}\n")

#  VISUALIZING BROADCASTING (The "Grid" Effect) 
# 'a' is a column (3, 1). 'b' is a row (2,). 
# To add them, PyTorch stretches 'a' horizontally (3, 2) and 'b' vertically (3, 2).
# This creates a 3x2 grid where every row in 'a' meets every column in 'b'.
a = torch.tensor([[1], [2], [3]])  # Shape: (3, 1)
b = torch.tensor([10, 20])          # Shape: (2,)
print(f"a shape: {a.shape}")
print(f"b shape: {b.shape}")
# Result:
# [1+10, 1+20]
# [2+10, 2+20]
# [3+10, 3+20]
print(f"a + b (broadcasts to (3, 2)):\n{a + b}")



## 4. Automatic Differentiation

Neural networks learn by adjusting their weights based on how "wrong" their predictions are. This process requires several key steps:

- **Forward pass:** Input → Model → Prediction
- **Calculate loss:** How far off was the prediction?
- **Backward pass:** Calculate **Gradients** (the direction to adjust weights)
- **Update weights:** Move the weights slightly to reduce the loss

PyTorch's **Autograd** engine automatically builds a "Computational Graph" to track these operations for you.

**The Math Memory:** `y.grad_fn`

When you perform math on tensors with `requires_grad=True`, PyTorch doesn't just store the answer; it stores the **Recipe** for how to reverse that math.

- **Definition:** `y.grad_fn` is a reference to the **Gradient Function** (the derivative) of the last operation performed.
- **The Chain:** If , the last operation was **Addition**. Therefore, `y.grad_fn` will be an `<AddBackward>` object. This object points back to the multiplication operation, creating a chain.


In [ ]:
# Create tensors with gradient tracking
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Forward pass: y = w * x + b
y = w * x + b

print(f"x = {x.item()}")
print(f"w = {w.item()}")
print(f"b = {b.item()}")
print(f"y = w * x + b = {y.item()}")
print(f"\nGradient function: {y.grad_fn}")

**Key Summary Table**

| Component | Purpose | Analogy |
|  |  |  |
| **`.item()`** | Returns the raw numerical value () | The finished cake |
| **`.grad_fn`** | Stores the derivative logic (`AddBackward`) | The recipe for the cake |
| **`.grad`** | Stores the actual slope/gradient (calculated after `.backward()`) | The direction to move |


### Computing Gradients with `.backward()`

In training, we don't just want to know the answer. We want to know how to change our inputs to get a _better_ answer next time.

**The Power of `.backward()`**

-   **Automated Calculus:** When you call `loss.backward()`, PyTorch applies the **Chain Rule** from calculus across the entire computational graph.
    
-   **The Destination:** The resulting "slopes" (gradients) are stored in the **`.grad`** attribute of any tensor that had `requires_grad=True`.
    
-   **The Goal:** A gradient tells you the direction and magnitude of the steepest increase. To _minimize_ loss, we move the weights in the **opposite** direction of the gradient.
    

**How the math works behind the scenes**

If our loss function is $L = (w \cdot x + b)^2$, PyTorch calculates the partial derivatives:

-   **For $w$:** $\frac{\partial L}{\partial w} = 2(w \cdot x + b) \cdot x$
    
-   **For $b$:** $\frac{\partial L}{\partial b} = 2(w \cdot x + b) \cdot 1$
    

In [ ]:
# Create tensors
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Forward pass
y = w * x + b
loss = y ** 2  # Some loss function

print(f"Loss: {loss.item()}\n")

# Backward pass - compute gradients
loss.backward()

# Gradients are stored in .grad attribute
print(f"Gradient of loss w.r.t. x: {x.grad}")
print(f"Gradient of loss w.r.t. w: {w.grad}")
print(f"Gradient of loss w.r.t. b: {b.grad}")

# Manual calculation to verify:
# loss = (w*x + b)^2 = (3*2 + 1)^2 = 49
# d(loss)/dx = 2*(w*x + b) * w = 2*7*3 = 42 ✓
# d(loss)/dw = 2*(w*x + b) * x = 2*7*2 = 28 ✓
# d(loss)/db = 2*(w*x + b) * 1 = 2*7 = 14 ✓

**Important Rules to Remember**

-   **Gradients Accumulate:** If you call `.backward()` twice without clearing the gradients, PyTorch will **add** the new gradients to the old ones. This is why you'll often see `optimizer.zero_grad()` in training loops.
    
-   **Memory Management:** Once `.backward()` is called, the internal graph is usually destroyed to save memory. If you need to call it again on the same graph, you would need `retain_graph=True`.


### Gradient Accumulation and Zeroing

**Important**: Gradients accumulate by default!

In PyTorch, the gradients are **accumulated** by default. This means every time you call `.backward()`, the newly calculated slopes are summed up with the existing values in the `.grad` buffer.

**Why does PyTorch do this?**

-   **Memory Efficiency:** It allows you to process very large datasets in small "mini-batches." You can sum the gradients from several small steps before actually updating your model weights, effectively simulating a larger batch size without needing a massive GPU.
    
-   **Complex Architectures:** Some models have multiple "heads" or loss functions that all need to contribute to the same set of weights.
    

**The "Zeroing" Requirement**

Because of this accumulation, you must explicitly **clear the gradients** before the next training step. If you forget, your model will "remember" gradients from previous iterations, causing the weights to jump in the wrong direction and preventing the model from learning.

In [ ]:
# Create a tensor
x = torch.tensor([1.0], requires_grad=True)

# First backward pass
y = x ** 2
y.backward()
print(f"First backward - Gradient: {x.grad}")

# Second backward pass (without zeroing)
y = x ** 2
y.backward()
print(f"Second backward - Gradient (accumulated!): {x.grad}")

# Zero the gradient
x.grad.zero_()
print(f"After zeroing: {x.grad}")

# Third backward pass
y = x ** 2
y.backward()
print(f"Third backward - Gradient: {x.grad}")

**Common Training Pattern**

In a real training loop, you will almost always see this sequence to handle the accumulation:

1.  `optimizer.zero_grad()` — Clear the old "memory."
    
2.  `loss.backward()` — Calculate new gradients.
    
3.  `optimizer.step()` — Update the weights.
   

**Why Do We Need to Zero Gradients?**

In PyTorch, gradients are **accumulated by default**.
Each time we call `.backward()`, the new gradient is added to the existing one:

$$
\text{grad}*{new} = \text{grad}*{old} + \frac{dL}{dx}
$$



**Small Mathematical Example (Same as the Code)**

Let:

$$
y = x^2
$$

Derivative:

$$
\frac{dy}{dx} = 2x
$$

If $x = 1$, then:

$$
\frac{dy}{dx} = 2
$$



**First backward**

Initially $x.grad = 0$

$$
x.grad = 0 + 2 = 2
$$



**Second backward (without zeroing)**

PyTorch computes $2$ again and adds it:

$$
x.grad = 2 + 2 = 4
$$

Now the gradient is incorrect for a single step.



**Why We Zero**

Before computing the next gradient, we reset:

$$
x.grad = 0
$$

Then the next backward gives:

$$
x.grad = 0 + 2 = 2
$$



**Key Idea**

Without zeroing:

$$
\text{gradient} = \text{old} + \text{new}
$$

With zeroing:

$$
\text{gradient} = \text{new only}
$$


### Disabling Gradient Tracking

During inference (prediction), we do not need derivatives like $\frac{dL}{dx}$.
So we turn off gradient tracking to save memory and computation.

**Why? (Mathematically)**

In training, we compute:

$$
\frac{dL}{dx}
$$

This requires storing intermediate values to apply the chain rule:

$$
\frac{dL}{dx} = \frac{dL}{dy} \cdot \frac{dy}{dx}
$$

That means PyTorch must build and store a **computational graph**.

But during inference, we only need:

$$
y = f(x)
$$

We do **not** compute:

$$
\frac{dy}{dx}
$$

So storing the graph is unnecessary.

**Method 1 — `torch.no_grad()`**

Inside this block:

```python
with torch.no_grad():
    y = x ** 2
```

Mathematically:

$$
y = x^2
$$

But PyTorch does **not** track:

$$
\frac{dy}{dx}
$$

So:

$$
y.requires_grad = False
$$

No graph is created.

**Method 2 — `.detach()`**

If:

$$
y = x^2
$$

Normally this tracks gradients.

When we do:

$$
y_{detached} = y.detach()
$$

We create a new tensor:

$$
y_{detached} = y
$$

but it is treated as a constant:

$$
\frac{dy_{detached}}{dx} = 0
$$

So:

$$
y_{detached}.requires_grad = False
$$

It is removed from the computation graph.


**Method 3 — Decorator**

```python
@torch.no_grad()
def predict(x):
    return x ** 2
```

This is just a cleaner way of writing:

$$
y = f(x)
$$

without tracking gradients.


**Key Idea**

Training:

$$
y = f(x), \quad \text{store graph}, \quad compute \frac{dL}{dx}
$$

Inference:

$$
y = f(x), \quad \text{no graph}, \quad \text{no derivatives}
$$

**Why This Is Important (Especially for LLMs)**

* Saves GPU memory
* Makes inference faster
* Prevents accidental gradient accumulation
* Required when serving models in production


**Mental Model**

Training = function + derivatives
Inference = function only

If you don’t need $\frac{dL}{dx}$, disable gradient tracking.


In [ ]:
x = torch.tensor([2.0], requires_grad=True)

# Method 1: torch.no_grad() context
with torch.no_grad():
    y = x ** 2
    print(f"Inside no_grad - requires_grad: {y.requires_grad}")

# Method 2: .detach() method
y = x ** 2
y_detached = y.detach()
print(f"Detached - requires_grad: {y_detached.requires_grad}")

# Method 3: Using decorators
@torch.no_grad()
def predict(x):
    return x ** 2

y = predict(x)
print(f"From function - requires_grad: {y.requires_grad}")

### Practice: Simple Linear Regression with Gradient Descent

I'll break down this linear regression example step by step so we understand exactly what's happening!

#### The Big Picture

You're training a model to find the line that best fits some data points. The true relationship is `y = 2x + 1`, but you're starting with random guesses and using gradient descent to find the correct values.

#### Step-by-Step Breakdown

1. **Generate Synthetic Data**

    ```python
    torch.manual_seed(42)  # Makes random numbers reproducible
    X = torch.randn(100, 1)  # 100 random x values
    y_true = 2 * X + 1 + 0.1 * torch.randn(100, 1)
    ```

    **What's happening:**
    - Creates 100 data points
    - True relationship: `y = 2x + 1` (slope=2, intercept=1)
    - Adds small noise (`0.1 * torch.randn(100, 1)`) to make it realistic
    - Example: if X[0] = 0.5, then y_true[0] ≈ 2.0

2. **Initialize Random Parameters**

    ```python
    w = torch.randn(1, requires_grad=True)  # Random weight (slope)
    b = torch.randn(1, requires_grad=True)  # Random bias (intercept)
    ```

    **What's happening:**
    - `w` is your guess for the slope (trying to find 2.0)
    - `b` is your guess for the intercept (trying to find 1.0)
    - `requires_grad=True` tells PyTorch: "I want to calculate gradients for these"
    - They start at random values (maybe w=0.5, b=-0.3)

3. **Training Loop Setup**

    ```python
    learning_rate = 0.01  # How big of steps to take
    num_epochs = 100      # How many times to go through the data
    losses = []           # Track how well we're doing
    ```

    **Learning rate:** Controls step size. Too big = unstable, too small = slow learning

4. **The Training Loop** (This is where the magic happens!)

    **Forward Pass** - Make predictions

    ```python
    y_pred = w * X + b
    ```

    **What's happening:**
    - Use current w and b to predict y
    - Example iteration 1: if w=0.5, b=-0.3, X[0]=0.5
    - y_pred[0] = 0.5 * 0.5 + (-0.3) = -0.05
    - But y_true[0] ≈ 2.0 (remember, true is 2*0.5 + 1 = 2.0)
    - We're way off! 😱

    **Compute Loss** - How wrong are we?

    ```python
    loss = ((y_pred - y_true) ** 2).mean()
    ```

    **What's happening:**
    - MSE (Mean Squared Error) = average of squared differences
    - For each point: (prediction - actual)²
    - Then average all 100 points
    - Example: if y_pred[0]=-0.05, y_true[0]=2.0
    - Error = (-0.05 - 2.0)² = (-2.05)² = 4.20
    - Higher loss = worse predictions

    **Backward Pass** - Calculate gradients

    ```python
    loss.backward()
    ```

    **This is the KEY magic!**

    **What PyTorch does automatically:**
    1. Looks at the computation graph: `loss = ((w*X + b) - y_true)².mean()`
    2. Uses chain rule to calculate:
    - `∂loss/∂w` (how much does changing w affect loss?)
    - `∂loss/∂b` (how much does changing b affect loss?)
    3. Stores these in `w.grad` and `b.grad`

    **Mathematical intuition:**
    - If gradient is positive → parameter is too high → decrease it
    - If gradient is negative → parameter is too low → increase it
    - Magnitude tells us how much to change

    **Update Parameters** - Take a step toward better values

    ```python
    with torch.no_grad():  # Don't track these operations for gradients
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    ```

    **What's happening:**
    - `w.grad` might be something like 5.3 (pointing upward)
    - `w = w - 0.01 * 5.3 = w - 0.053`
    - We move w in the OPPOSITE direction of the gradient (to go downhill)
    - Same for b

    **Example progression:**
    ```
    Epoch 1:  w=0.5000, b=-0.3000, loss=15.2345
    Epoch 20: w=1.8234, b=0.8912, loss=0.4523
    Epoch 40: w=1.9567, b=0.9734, loss=0.0234
    Epoch 80: w=1.9891, b=0.9956, loss=0.0012
    Epoch 100: w=1.9978, b=0.9989, loss=0.0003
    ```

    See how w→2.0 and b→1.0? 🎉

    **Zero Gradients**

    ```python
    w.grad.zero_()
    b.grad.zero_()
    ```

    **Why needed:**
    - PyTorch ACCUMULATES gradients by default
    - If you don't zero them, next iteration adds to previous gradients
    - Would cause wrong updates!

#### Visual Understanding

```
Initial random guess:
     y
     |     
   5 |           * True line: y=2x+1
     |         /
   3 |       / *
     |     /   
   1 |   /     * * Your line (wrong!)
     | /   *
  -1 |/________ x
    -1  0  1  2

After 100 epochs:
     y
     |     
   5 |           *
     |         /
   3 |       / * Your line ≈ True line!
     |     /     (w≈2, b≈1)
   1 |   /     *
     | /   *
  -1 |/________ x
    -1  0  1  2
```

#### What's Being Optimized

You're solving this optimization problem:

```
minimize: loss = (1/100) Σ[(w*x_i + b - y_i)²]

by adjusting: w and b

method: gradient descent
  - Calculate ∂loss/∂w and ∂loss/∂b
  - Update: w = w - lr * ∂loss/∂w
  - Update: b = b - lr * ∂loss/∂b
  - Repeat until convergence
```

#### Key Concepts

1. **Forward pass:** Use current parameters to make predictions
2. **Loss computation:** Measure how bad predictions are
3. **Backward pass:** Calculate gradients (how to improve)
4. **Parameter update:** Actually improve parameters
5. **Repeat:** Until loss is small enough

#### Why It Works

The loss function is like a bowl 🥣. You start at a random point on the bowl's surface. Gradients point uphill, so you walk downhill (negative gradient). Eventually you reach the bottom (minimum loss) where w≈2 and b≈1.

In [ ]:
# Generate synthetic data: y = 2x + 1 + noise
torch.manual_seed(42)
X = torch.randn(100, 1)
y_true = 2 * X + 1 + 0.1 * torch.randn(100, 1)

# Initialize random parameters (trainable -- requires_grad=True)
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print("initial w = ", w)
print("initial b = ", b)

# Hyperparameters
learning_rate = 0.01
num_epochs = 100

losses = []

# YOUR CODE HERE
#
# Implement gradient descent. For each of `num_epochs` iterations:
#
#   - Compute a forward pass: predict y from the current w, b, and the
#     input X using the linear model y = w * X + b.
#   - Compute the mean squared error loss between the predictions and y_true.
#   - Record the scalar loss value into the `losses` list (extract the
#     Python float from the tensor first).
#   - Run a backward pass to populate gradients on w and b.
#   - Update w and b in place using gradient descent (subtract learning_rate
#     times the parameter's gradient). The update itself should run inside
#     a no-gradient context so it is not tracked by autograd.
#   - Zero out the gradients on w and b in place so they don't carry over
#     into the next iteration.
#   - Every 20 epochs, print one status line showing the current epoch
#     number, the loss value, and the current values of w and b.
#
# After the loop finishes, w should be close to 2.0 and b close to 1.0
# (the true data-generating values). Print a final summary line confirming
# the values you converged to.


# Plot loss curve (uncomment once the training loop is filled in)
# plt.figure(figsize=(10, 4))
# plt.plot(losses)
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss Over Time')
# plt.grid(True)
# plt.show()


## 5. Building Neural Networks

### Building Neural Networks

#### The nn.Module Class

All neural networks in PyTorch inherit from `nn.Module`. This provides:
- Parameter management
- GPU support
- Saving/loading functionality
- Training/evaluation modes

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SimpleNN(nn.Module):
    """A small fully-connected feed-forward network.

    Architecture: input -> Linear -> ReLU -> Linear -> output.
    The first Linear maps input_size features to hidden_size features;
    the second maps hidden_size features to output_size features.
    A ReLU activation sits between the two Linear layers.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # YOUR CODE HERE
        #
        # Create two Linear layers as attributes on self:
        #   - one that maps input_size features to hidden_size features
        #   - one that maps hidden_size features to output_size features
        pass

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Pass x through the first Linear layer, apply a ReLU activation,
        # then pass the result through the second Linear layer. Return
        # the final output (no activation on the output layer).
        pass


**What's happening:**
- `fc1`: First layer transforms 10 inputs → 20 hidden features
  - Parameters: (10 × 20) weights + 20 biases = 220 parameters
- `fc2`: Second layer transforms 20 hidden → 5 outputs
  - Parameters: (20 × 5) weights + 5 biases = 105 parameters
- **Total:** 220 + 105 = 325 parameters

**Key components:**
- `__init__`: Define all layers (what the network contains)
- `forward`: Define how data flows through the network
- `super().__init__()`: Required to initialize the parent `nn.Module` class
- `F.relu()`: Activation function adds non-linearity between layers

#### Forward Pass Explained

```python
def forward(self, x):
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x
```

This function defines how the input `x` is transformed into the output of the network.

##### Step 1: First Linear Layer

```
x = self.fc1(x)
```

Mathematically:

$$
z_1 = xW_1^T + b_1
$$

* Multiplies input by weight matrix
* Adds bias
* Transforms dimension: `input_size → hidden_size`

##### Step 2: ReLU Activation

```
x = F.relu(...)
```

Applies the activation function:

$$
\text{ReLU}(t) = \max(0, t)
$$

So:

$$
a_1 = \text{ReLU}(z_1)
$$

This introduces non-linearity into the model.

Without ReLU, the entire network would behave like a single linear transformation.

##### Step 3: Second Linear Layer

```
x = self.fc2(x)
```

Mathematically:

$$
y = a_1 W_2^T + b_2
$$

* Maps `hidden_size → output_size`
* Produces the final output

##### Final Function Represented by the Network

$$
y = \big(\text{ReLU}(xW_1^T + b_1)\big) W_2^T + b_2
$$

This is the complete computation performed during the forward pass.


### Common Layer Types

Understanding these is essential for building LLMs:

**Linear (Fully Connected) Layer**



In [ ]:
# Linear (Fully Connected) Layer
linear = nn.Linear(in_features=512, out_features=256)
x = torch.randn(32, 512)  # Batch of 32 samples, 512 features each
output = linear(x)
print(f"Linear layer output shape: {output.shape}\n")

**What's happening:**

- Takes 512-dimensional input vectors

- Transforms each to 256-dimensional output via learned weights

- Formula: `output = x @ W.T + b` where W is (256, 512) and b is (256,)

  


**Embedding Layer (critical for LLMs!)**

In [ ]:
# Embedding Layer (critical for LLMs!)
vocab_size = 10000
embedding_dim = 512
embedding = nn.Embedding(vocab_size, embedding_dim)

# Token indices (like word IDs)
token_ids = torch.randint(0, vocab_size, (32, 50))  # 32 sentences, 50 tokens each
embedded = embedding(token_ids)
print(f"Embedding output shape: {embedded.shape}")  # (32, 50, 512)
print("  → 32 batches, 50 tokens, each token is a 512-dim vector\n")

**What's happening:**

- Creates a lookup table: 10,000 possible tokens, each mapped to 512-dim vector

- Input: token IDs (integers from 0 to 9,999)

- Output: dense vector representations

- Example: token ID 42 → [0.23, -0.45, 0.67, ..., 0.12] (512 values)

**LayerNorm (used in transformers)**

In [ ]:
# LayerNorm (used in transformers)
layer_norm = nn.LayerNorm(512)
normalized = layer_norm(embedded)
print(f"After LayerNorm: {normalized.shape}\n")

**What's happening:**

- Normalizes each 512-dim vector to have mean=0 and std=1

- Helps training stability in deep networks

- Shape unchanged, but values are normalized


In [ ]:
# Dropout (regularization)
dropout = nn.Dropout(p=0.1)  # Drop 10% of neurons
dropped = dropout(normalized)
print(f"After Dropout: {dropped.shape}")

**What's happening:**

- During training: randomly sets 10% of values to zero

- During evaluation: keeps all values (no dropout)

- Prevents overfitting by forcing network to be robust

- Shape unchanged, but some values are zeroed out

### Activation Functions

Activation functions add non-linearity, allowing networks to learn complex patterns:

In [ ]:
x = torch.linspace(-3, 3, 100)

plt.figure(figsize=(15, 4))

# ReLU (most common)
plt.subplot(1, 4, 1)
plt.plot(x.numpy(), F.relu(x).numpy())
plt.title('ReLU')
plt.grid(True)

# GELU (used in transformers)
plt.subplot(1, 4, 2)
plt.plot(x.numpy(), F.gelu(x).numpy())
plt.title('GELU')
plt.grid(True)

# Tanh
plt.subplot(1, 4, 3)
plt.plot(x.numpy(), torch.tanh(x).numpy())
plt.title('Tanh')
plt.grid(True)

# Sigmoid
plt.subplot(1, 4, 4)
plt.plot(x.numpy(), torch.sigmoid(x).numpy())
plt.title('Sigmoid')
plt.grid(True)

plt.tight_layout()
plt.show()

# Usage in networks
x_data = torch.randn(5, 10)
print(f"ReLU: {F.relu(x_data)[0, :3]}")
print(f"GELU: {F.gelu(x_data)[0, :3]}")

### Using nn.Sequential for Simple Models

In [ ]:
# Creating a model with Sequential
model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10)
)

print(model)

# Test forward pass
x = torch.randn(32, 784)  # 32 images, flattened to 784 pixels
output = model(x)
print(f"\nOutput shape: {output.shape}")


**What's happening:**
- `nn.Sequential` chains layers together in order
- Data flows: 784 → 256 → ReLU → Dropout → 128 → ReLU → Dropout → 10
- Example use case: MNIST digit classification (28×28 pixels = 784 inputs, 10 digit classes)

**Layer-by-layer transformation:**
```
Input:     [32, 784]  ← 32 flattened images
  ↓ Linear(784→256)
Hidden 1:  [32, 256]  ← First hidden layer
  ↓ ReLU (activation)
           [32, 256]  ← Non-linearity applied
  ↓ Dropout (20%)
           [32, 256]  ← 20% of values zeroed during training
  ↓ Linear(256→128)
Hidden 2:  [32, 128]  ← Second hidden layer
  ↓ ReLU
           [32, 128]
  ↓ Dropout (20%)
           [32, 128]
  ↓ Linear(128→10)
Output:    [32, 10]   ← 10 class scores for each image
```

**When to use Sequential:**
- ✅ Simple feedforward architectures
- ✅ Layers connect in straightforward order
- ❌ Complex architectures (skip connections, multiple inputs)
- ❌ When you need custom logic in forward pass

### Model Inspection

In [ ]:
def count_parameters(model):
    """Return the total number of trainable parameters in `model`.

    Iterate over every parameter in the model, keep only those whose
    `requires_grad` attribute is True (this excludes frozen layers), and
    sum the number of elements in each parameter tensor. Return the total
    count as an integer.
    """
    # YOUR CODE HERE
    pass


# After implementing count_parameters, uncomment this block to verify it:
# print(f"Total trainable parameters: {count_parameters(model):,}\n")
# print("All layers:")
# for name, param in model.named_parameters():
#     print(f"  {name:20s} | shape: {tuple(param.shape)} | params: {param.numel():,}")


**What's happening:**
- `p.numel()`: Returns number of elements in parameter tensor
- `requires_grad=True`: Only count trainable parameters (excludes frozen layers)
- Total: 325 parameters that will be updated during training

**Parameter breakdown:**
```
Layer 1 (fc1): Linear(10 → 20)
  ├─ fc1.weight: [20, 10] = 200 parameters  (weight matrix)
  └─ fc1.bias:   [20]     = 20 parameters   (bias vector)
                            ───
                            220 parameters

Layer 2 (fc2): Linear(20 → 5)
  ├─ fc2.weight: [5, 20]  = 100 parameters  (weight matrix)
  └─ fc2.bias:   [5]      = 5 parameters    (bias vector)
                            ───
                            105 parameters

Total: 220 + 105 = 325 parameters
```

**Key concepts:**
- **Weight matrices**: Store learned connections between neurons
- **Bias vectors**: One bias per output neuron
- **Shape convention**: Linear layer weights are `[out_features, in_features]`
- **Parameter counting**: Essential for understanding model size and memory requirements

**Example calculation:**
```python
# For fc1.weight with shape [20, 10]:
out_features = 20  # Number of neurons in this layer
in_features = 10   # Number of neurons from previous layer
total_weights = out_features × in_features = 20 × 10 = 200
```




## 6. Training Neural Networks

### The Training Loop Pattern

All PyTorch training follows this pattern:

```python
for epoch in range(num_epochs):
    for batch in dataloader:
        # 1. Forward pass
        predictions = model(inputs)
        
        # 2. Compute loss
        loss = loss_function(predictions, targets)
        
        # 3. Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # 4. Update weights
        optimizer.step()
```

**What's happening:**

1.  **Forward pass**: Feed data through the model to get predictions
2.  **Compute loss**: Measure how wrong the predictions are
3.  **Backward pass**: Calculate gradients (how to improve)
    -   `optimizer.zero_grad()`: Clear old gradients (prevents accumulation)
    -   `loss.backward()`: Compute new gradients via backpropagation
4.  **Update weights**: Adjust parameters using gradients
    -   `optimizer.step()`: Apply gradient descent update


### Creating a Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader


class SimpleDataset(Dataset):
    """A minimal custom Dataset that wraps two tensors X and y.

    Implement three methods on the class:

    - __init__: store X and y as attributes on self so they can be accessed
      later by __getitem__.

    - __len__: return the total number of samples in the dataset (the length
      of X, which is the same as the length of y).

    - __getitem__: given an integer index, return a tuple of (one sample
      from X at that index, one sample from y at that index).
    """
    # YOUR CODE HERE
    pass


# After implementing SimpleDataset, uncomment this block to verify it works:
# X_train = torch.randn(1000, 20)
# y_train = torch.randint(0, 2, (1000,))
# dataset = SimpleDataset(X_train, y_train)
# print(f"Dataset length: {len(dataset)}")
#
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
# for batch_X, batch_y in dataloader:
#     print(f"Batch X shape: {batch_X.shape}, Batch y shape: {batch_y.shape}")
#     break


**Understanding Dataset and DataLoader:**

**Dataset** - A class that knows how to access your data:

-   Must implement 3 methods:
    -   `__init__`: Initialize with data (X and y)
    -   `__len__`: Return total number of samples (e.g., 1000)
    -   `__getitem__`: Return a single sample given an index

```python
# How Dataset works internally:
dataset = SimpleDataset(X_train, y_train)
print(len(dataset))        # Calls __len__() → 1000
sample = dataset[5]        # Calls __getitem__(5) → (X_train[5], y_train[5])

```

**DataLoader** - Handles batching, shuffling, and parallel loading:

-   **batch_size=32**: Group 32 samples together
-   **shuffle=True**: Randomly reorder data each epoch (prevents learning order)
-   **shuffle=False**: Keep order (for test/validation data)
-   **num_workers**: Number of parallel processes for loading (default=0)

**Batch calculation:**

```
1000 samples ÷ 32 batch_size = 31.25
→ 31 full batches of 32 samples
→ 1 final batch of 8 samples
→ Total: 32 batches

```

**Why batching matters:**

-   **Memory efficient**: Can't fit all 1000 samples in GPU at once
-   **Faster training**: Parallel matrix operations on batches
-   **Better gradients**: Average gradient over batch is more stable than single sample


### Loss Functions

Common loss functions for different tasks:

**Binary Classification (2 classes)**



In [ ]:
# Binary Classification (2 classes)
predictions_binary = torch.sigmoid(torch.randn(10, 1))
targets_binary = torch.randint(0, 2, (10, 1)).float()
loss_binary = F.binary_cross_entropy(predictions_binary, targets_binary)
print(f"Binary Cross Entropy: {loss_binary.item():.4f}\n")


**What's happening:**

-   **Predictions**: Probabilities between 0 and 1 (after sigmoid)
-   **Targets**: Actual labels (0 or 1)
-   **BCE Loss**: Measures how far predictions are from true labels
-   **Use case**: Spam detection, yes/no decisions



**Multi-class Classification (3+ classes)**



In [ ]:
# Multi-class Classification (3+ classes)
predictions_multi = torch.randn(10, 5)  # 10 samples, 5 classes (logits)
targets_multi = torch.randint(0, 5, (10,))  # Target class indices
loss_multi = F.cross_entropy(predictions_multi, targets_multi)
print(f"Cross Entropy: {loss_multi.item():.4f}\n")

**What's happening:**

-   **Logits**: Raw scores (not probabilities), shape [10, 5]
-   **Targets**: Class indices (0, 1, 2, 3, or 4), shape [10]
-   **CE Loss**: Applies softmax internally, then computes loss
-   **Use case**: Image classification (cat/dog/bird), digit recognition

**Important:** Cross entropy expects raw logits, NOT probabilities!

**Regression**


In [ ]:
# Regression
predictions_reg = torch.randn(10, 1)
targets_reg = torch.randn(10, 1)
loss_mse = F.mse_loss(predictions_reg, targets_reg)
print(f"Mean Squared Error: {loss_mse.item():.4f}")

loss_mae = F.l1_loss(predictions_reg, targets_reg)
print(f"Mean Absolute Error: {loss_mae.item():.4f}")

**What's happening:**

-   **MSE (L2 Loss)**: Average of (prediction - target)²
    -   Penalizes large errors heavily
    -   Sensitive to outliers
-   **MAE (L1 Loss)**: Average of |prediction - target|
    -   Penalizes errors linearly
    -   More robust to outliers
-   **Use case**: Predicting house prices, temperature, stock prices



### Optimizers

Optimizers update model parameters based on gradients:

In [ ]:
model = SimpleNN(20, 50, 2)

# SGD (Stochastic Gradient Descent)
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam (Adaptive Moment Estimation) - most popular!
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.001)

# AdamW (Adam with weight decay) - used in transformers
optimizer_adamw = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print("Optimizers created!")
print(f"\nAdam optimizer state: {optimizer_adam}")

**Understanding Optimizers:**

**SGD (Stochastic Gradient Descent)**

```python
# Basic update rule:
w = w - lr * gradient

# With momentum (smooths updates):
velocity = momentum * velocity - lr * gradient
w = w + velocity

```

-   **lr=0.01**: Learning rate (step size)
-   **momentum=0.9**: Uses 90% of previous update direction
-   **Pros**: Simple, well-understood
-   **Cons**: Sensitive to learning rate, slow on some problems

**Adam (Adaptive Moment Estimation)**

```python
# Maintains running averages:
m = beta1 * m + (1-beta1) * gradient        # First moment (mean)
v = beta2 * v + (1-beta2) * gradient²       # Second moment (variance)
w = w - lr * m / sqrt(v)                     # Adaptive step

```

-   **lr=0.001**: Learning rate (often smaller than SGD)
-   **betas=(0.9, 0.999)**: Decay rates for moment estimates
-   **Pros**: Adapts learning rate per parameter, works well by default
-   **Cons**: Can overfit, uses more memory

**AdamW (Adam with Weight Decay)**

```python
# Same as Adam, but decouples weight decay:
w = w - lr * (gradient + weight_decay * w)

```

-   **weight_decay=0.01**: L2 regularization strength
-   **Pros**: Better generalization, preferred for transformers/LLMs
-   **Use case**: Large language models (GPT, BERT, etc.)

**Optimizer comparison:**

```
Task                  | Recommended Optimizer
─────────────────────|──────────────────────
Vision (CNNs)        | SGD with momentum
General deep learning| Adam
Transformers/LLMs    | AdamW
Research/prototyping | Adam (good default)
Production (fine-tune)| AdamW

```

### Complete Training Example

**Setup: Model, Loss, and Optimizer**

```python
# Create model
model = SimpleNN(input_size=20, hidden_size=50, output_size=2)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

```

**What's happening:**

-   **Model**: 20 inputs → 50 hidden → 2 outputs (binary classification)
-   **Criterion**: Cross-entropy loss for classification
-   **Optimizer**: Adam with learning rate 0.001

In [ ]:
# ---------------------------------------------------------------
# Data setup -- create train_loader and test_loader from synthetic data
# ---------------------------------------------------------------
# YOUR CODE HERE
#
# 1. Set a manual seed so this notebook is reproducible.
#
# 2. Generate a synthetic feature tensor of shape (1000, 20) using random
#    normal values, and a synthetic label tensor of shape (1000,) with
#    random integers in {0, 1} (binary classification).
#
# 3. Split the features and labels into a training portion (first 800 samples)
#    and a test portion (remaining 200).
#
# 4. Wrap each (X, y) split in a SimpleDataset (the class you defined earlier
#    in this notebook). Call them train_dataset and test_dataset.
#
# 5. Wrap each dataset in a DataLoader with batch size 32. Shuffle the
#    training loader; do NOT shuffle the test loader (evaluation should
#    be deterministic). Name them train_loader and test_loader.


# ---------------------------------------------------------------
# Model, loss, optimizer
# ---------------------------------------------------------------
model = SimpleNN(input_size=20, hidden_size=50, output_size=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# ---------------------------------------------------------------
# Training function
# ---------------------------------------------------------------
def train_epoch(model, dataloader, criterion, optimizer):
    """One full pass over `dataloader`. Returns (avg_loss, accuracy_pct).

    For each batch of (X_batch, y_batch) from the dataloader:
      - Run a forward pass through the model to get predictions.
      - Compute the loss from predictions and y_batch using `criterion`.
      - Zero gradients, do a backward pass, and step the optimizer.
      - Accumulate the loss value into a running total (use .item()).
      - From predictions, pick the class with the highest score per sample.
      - Count how many of those match y_batch and add to a running correct count.
      - Add the batch size to a running total-sample count.

    After the loop, compute and return:
      - average loss = running total loss / number of batches
      - accuracy percent = 100 * correct count / total samples
    """
    model.train()
    # YOUR CODE HERE
    pass


# ---------------------------------------------------------------
# Evaluation function
# ---------------------------------------------------------------
def evaluate(model, dataloader, criterion):
    """Same metrics as train_epoch, but with no gradient updates.

    Wrap the forward pass in a no-grad context so PyTorch does not build
    a computation graph. Do NOT call zero_grad, backward, or optimizer.step.
    The rest of the logic (loss accumulation, accuracy counting, averaging)
    is identical to train_epoch.

    Returns the same (avg_loss, accuracy_pct) pair.
    """
    model.eval()
    # YOUR CODE HERE
    pass


# ---------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------
num_epochs = 20
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# YOUR CODE HERE
#
# Run a loop over num_epochs. In each iteration:
#   - Call train_epoch with the model, train_loader, criterion, optimizer.
#     Capture the returned (train_loss, train_acc).
#   - Call evaluate with the model, test_loader, criterion.
#     Capture the returned (test_loss, test_acc).
#   - Append all four values to their respective lists declared above.
#   - Every 5 epochs, print one status line showing the epoch number,
#     train loss, train accuracy, test loss, and test accuracy.


# ---------------------------------------------------------------
# Plot results (uncomment once the training loop is filled in)
# ---------------------------------------------------------------
# import matplotlib.pyplot as plt
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
# ax1.plot(train_losses, label='Train'); ax1.plot(test_losses, label='Test')
# ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True)
# ax2.plot(train_accs, label='Train');   ax2.plot(test_accs,  label='Test')
# ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.legend(); ax2.grid(True)
# plt.tight_layout(); plt.show()


**Training Function**

**What's happening:**

**`model.train()`** - Sets model to training mode:

-   Enables dropout (randomly drops neurons)
-   Enables batch normalization updates
-   Important: Affects model behavior!

**Forward pass:**

```python
predictions = model(X_batch)  # Shape: [32, 2] (32 samples, 2 class scores)
loss = criterion(predictions, y_batch)  # Single number (average loss)

```

**Backward pass:**

```python
optimizer.zero_grad()  # Clear old gradients
loss.backward()        # Compute new gradients
optimizer.step()       # Update weights: w = w - lr * gradient

```

**Tracking metrics:**

```python
_, predicted = predictions.max(1)  # Get class with highest score
# predictions = [[2.3, -1.1],    →  predicted = [0,
#                [-0.5, 1.8]]                      1]
# (picks index of max value per row)

correct += predicted.eq(y_batch).sum().item()  # Count correct predictions
accuracy = 100.0 * correct / total              # Percentage correct

```

**Evaluation Function**

**Key differences from training:**

**`model.eval()`** - Sets model to evaluation mode:

-   Disables dropout (keeps all neurons)
-   Uses fixed batch normalization statistics
-   Makes predictions deterministic

**`with torch.no_grad()`** - Disables gradient tracking:

-   Saves memory (no need to store computation graph)
-   Speeds up inference (no backward pass needed)
-   Essential for evaluation!

**Why we don't need gradients during evaluation:**

```
Training:   data → model → loss → gradients → update weights
Evaluation: data → model → loss → [done]  (no weight updates)

```

**Main Training Loop**

**What's happening:**

**Per epoch:**

1.  Train on all training batches (32 batches)
2.  Evaluate on all test batches (7 batches)
3.  Store metrics for plotting
4.  Print progress every 5 epochs

**Reading the output:**

-   **Train Loss decreasing**: Model is learning (0.2666 → 0.0462)
-   **Train Acc increasing**: Getting better at training data (95.4% → 99.8%)
-   **Test Loss decreasing**: Generalizing to new data (0.2626 → 0.0854)
-   **Test Acc increasing**: Good performance on unseen data (93.0% → 97.0%)

**Healthy training signs:**

-  Both train and test metrics improving
-  Test accuracy following train accuracy closely
-  No huge gap between train and test (not overfitting)

**What to look for in plots:**

**Loss plot (left):**

```
Loss
 │
 │ \
 │  \___  Train Loss (should decrease smoothly)
 │   \___
 │    \__ Test Loss (should follow train closely)
 │     \__
 └─────────── Epochs

```

**Accuracy plot (right):**

```
Acc
 │      ___
 │    /    Train Acc (should increase)
 │  /____  
 │ /      Test Acc (should follow train)
 │/
 └─────────── Epochs

```

**Diagnostic patterns:**

**Good training (what we see):**

-   Both losses decreasing
-   Both accuracies increasing
-   Small gap between train/test

**Overfitting:**

-   Train loss ↓, test loss ↑
-   Train acc ↑, test acc ↓
-   Large gap between train/test

**Underfitting:**

-   Both losses high and not decreasing
-   Both accuracies low and not improving
-   Model too simple or learning rate too low

**Complete training workflow summary:**

```
1. Prepare data (Dataset + DataLoader)
2. Create model (nn.Module)
3. Choose loss function (criterion)
4. Choose optimizer (Adam, SGD, etc.)
5. Training loop:
   ├─ Train one epoch (forward → loss → backward → update)
   ├─ Evaluate on test set
   └─ Track and plot metrics
6. Analyze results (loss curves, accuracy)

```

### Saving and Loading Models

In [ ]:
# Save model
torch.save(model.state_dict(), 'model.pth')
print("Model saved!\n")

# Load model
loaded_model = SimpleNN(input_size=20, hidden_size=50, output_size=2)
loaded_model.load_state_dict(torch.load('model.pth', weights_only=True))
loaded_model.eval()
print("Model loaded!\n")

# Verify it works
test_input = torch.randn(1, 20)
with torch.no_grad():
    output = loaded_model(test_input)
    print(f"Test output: {output}")

# Save full checkpoint (model + optimizer state)
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'test_loss': test_losses[-1]
}
torch.save(checkpoint, 'checkpoint.pth')
print("\nCheckpoint saved!")

**Save Model Weights**

```python
# Save model
torch.save(model.state_dict(), 'model.pth')
print("Model saved!\n")

```

**Output:**

```
Model saved!

```

**What's happening:**

-   `model.state_dict()`: Returns a dictionary of all model parameters
    
    ```python
    {    'fc1.weight': tensor([[...]]),  # Shape: [50, 20]    'fc1.bias': tensor([...]),      # Shape: [50]    'fc2.weight': tensor([[...]]),  # Shape: [2, 50]    'fc2.bias': tensor([...])       # Shape: [2]}
    
    ```
    
-   `torch.save()`: Serializes and saves to disk
-   `.pth` extension: PyTorch convention (could use `.pt` or `.bin`)
-   **Only saves weights**, not the model architecture

----------

**Load Model Weights**

```python
# Load model
loaded_model = SimpleNN(input_size=20, hidden_size=50, output_size=2)
loaded_model.load_state_dict(torch.load('model.pth', weights_only=True))
loaded_model.eval()
print("Model loaded!\n")

```

**Output:**

```
Model loaded!

```

**What's happening:**

**Step 1: Create model architecture**

```python
loaded_model = SimpleNN(input_size=20, hidden_size=50, output_size=2)
# Creates model with RANDOM weights initially

```

**Step 2: Load saved weights**

```python
loaded_model.load_state_dict(torch.load('model.pth', weights_only=True))
# Replaces random weights with trained weights

```

**Step 3: Set to evaluation mode**

```python
loaded_model.eval()
# Disables dropout, batch norm training mode

```

**Important:**

-   `weights_only=True`: Security feature (prevents arbitrary code execution)
-   Must create model with **exact same architecture** before loading
-   Architecture not saved, only the learned parameters

----------

**Verify Model Works**

```python
# Verify it works
test_input = torch.randn(1, 20)
with torch.no_grad():
    output = loaded_model(test_input)
    print(f"Test output: {output}")

```

**Output:**

```
Test output: tensor([[ 4.1292, -4.1042]])

```

**What's happening:**

-   Creates random input: shape [1, 20] (1 sample, 20 features)
-   `with torch.no_grad()`: Disables gradient tracking (faster, less memory)
-   Output: [4.1292, -4.1042]
    -   Two scores for binary classification
    -   Class 0 score: 4.1292 (higher)
    -   Class 1 score: -4.1042 (lower)
    -   Model predicts class 0

**To get predicted class:**

```python
_, predicted_class = output.max(1)  # Returns index of max value
# predicted_class = 0

```

----------

**Save Complete Checkpoint**

```python
# Save full checkpoint (model + optimizer state)
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'test_loss': test_losses[-1]
}
torch.save(checkpoint, 'checkpoint.pth')
print("\nCheckpoint saved!")

```

**Output:**

```
Checkpoint saved!

```

**What's happening:**

**Checkpoint contains everything needed to resume training:**

```python
checkpoint = {
    'epoch': 20,                           # Which epoch we finished
    'model_state_dict': {...},             # Model weights
    'optimizer_state_dict': {...},         # Optimizer state (momentum, etc.)
    'train_loss': 0.0462,                  # Final training loss
    'test_loss': 0.0854                    # Final test loss
}

```

**Why save optimizer state?**

-   Optimizers like Adam maintain internal state:
    -   Running averages of gradients
    -   Running averages of squared gradients
    -   Step counts
-   Without this state, resumed training won't continue smoothly

**Loading a checkpoint to resume training:**

```python
# Load checkpoint
checkpoint = torch.load('checkpoint.pth', weights_only=False)

# Restore model
model = SimpleNN(20, 50, 2)
model.load_state_dict(checkpoint['model_state_dict'])

# Restore optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Resume training from saved epoch
start_epoch = checkpoint['epoch']
print(f"Resuming from epoch {start_epoch}")

# Continue training loop
for epoch in range(start_epoch, start_epoch + 10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    # ... rest of training

```

----------

**Comparison: state_dict vs Checkpoint**

```
┌─────────────────────────────────────────────────────────────────┐
│                  model.state_dict()                             │
├─────────────────────────────────────────────────────────────────┤
│ ✓ Model weights only                                            │
│ ✓ Smallest file size                                            │
│ ✓ Use for inference/deployment                                  │
│ ✗ Cannot resume training exactly                                │
│                                                                 │
│ Use case: Deploy trained model to production                    │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    Full Checkpoint                              │
├─────────────────────────────────────────────────────────────────┤
│ ✓ Model weights                                                 │
│ ✓ Optimizer state                                               │
│ ✓ Training metadata (epoch, losses)                             │
│ ✓ Can resume training exactly                                   │
│ ✗ Larger file size                                              │
│                                                                 │
│ Use case: Long training runs, continue interrupted training     │
└─────────────────────────────────────────────────────────────────┘

```

**Best practices:**

**During training:**

```python
# Save checkpoint every N epochs
if (epoch + 1) % 10 == 0:
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'best_test_acc': best_test_acc
    }, f'checkpoint_epoch_{epoch+1}.pth')

```

**After training:**

```python
# Save best model based on validation accuracy
if test_acc > best_test_acc:
    best_test_acc = test_acc
    torch.save(model.state_dict(), 'best_model.pth')

```

**For deployment:**

```python
# Load only weights for inference
model = SimpleNN(20, 50, 2)
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
model.eval()  # Set to evaluation mode

```

**File size example:**

```
model.pth (weights only):        ~50 KB
checkpoint.pth (full):           ~100 KB
(includes optimizer state, doubles size)

```



## 7. Working with GPUs

### Moving Tensors and Models to GPU

In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Memory cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

**What's happening:**

**Device selection:**

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# If GPU available: device = torch.device('cuda')
# If no GPU:        device = torch.device('cpu')

```

**Why this pattern is useful:**

-   Code runs on both GPU and CPU without changes
-   Automatically uses GPU when available
-   Falls back to CPU if no GPU

**GPU information:**

-   `get_device_name(0)`: Name of GPU 0 (first GPU)
-   `memory_allocated()`: GPU memory actively used by tensors
-   `memory_reserved()`: GPU memory reserved by PyTorch (may be unused)

**Multiple GPU indexing:**

```python
cuda:0  →  First GPU
cuda:1  →  Second GPU
cuda:2  →  Third GPU

```

**Move tensors to GPU**

In [ ]:
# Move tensor to GPU
tensor_cpu = torch.randn(3, 3)
print(f"CPU tensor device: {tensor_cpu.device}")

tensor_gpu = tensor_cpu.to(device)
print(f"GPU tensor device: {tensor_gpu.device}")



**What's happening:**

**`.to(device)` method:**

-   Creates a copy of the tensor on the target device
-   Original tensor remains on original device
-   Returns new tensor on target device

**Important rule:** All tensors in an operation must be on the same device!

```python
# ✗ This will ERROR:
cpu_tensor = torch.randn(3, 3)
gpu_tensor = torch.randn(3, 3).to('cuda')
result = cpu_tensor + gpu_tensor  # RuntimeError: Expected all tensors on same device

# ✓ This works:
cpu_tensor = torch.randn(3, 3)
gpu_tensor = cpu_tensor.to('cuda')
result = gpu_tensor + gpu_tensor  # Both on GPU

```


**Move model to GPU**



In [ ]:
# Move model to GPU
model = SimpleNN(20, 50, 2)
model = model.to(device)
print(f"\nModel device: {next(model.parameters()).device}")

**What's happening:**

**`model.to(device)` moves ALL model components:**

-   All weight matrices
-   All bias vectors
-   All internal buffers (batch norm statistics, etc.)

**Checking model device:**

```python
next(model.parameters()).device
# Gets first parameter's device
# All parameters should be on same device

```

**In-place vs. copy:**

```python
# In-place (modifies model directly)
model.to(device)  # model is now on GPU

# Assignment (common pattern for clarity)
model = model.to(device)  # Explicit that model moved

```


### GPU-Accelerated Training

Just add `.to(device)` to tensors and model!

In [ ]:
# Setup -- create model and optimizer on the chosen device
model = SimpleNN(20, 50, 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# YOUR CODE HERE
#
# Implement a 3-epoch GPU-aware training loop.
#
# PREREQUISITE: this cell depends on `train_loader` defined in the earlier
# "complete training example" cell. Make sure you have run that cell first
# so train_loader exists in this notebook's namespace.
#
# This is the same shape as the CPU training loop you wrote earlier, with
# one critical difference: before passing each batch into the model, move
# both the input batch and the label batch onto `device`. The model itself
# was already moved with .to(device) above.
#
# Inside each iteration, follow the standard four-step training rhythm:
# forward pass, loss computation, then zero_grad / backward / step.
#
# At the end of each epoch (after the inner batch loop completes), print
# one status line showing the epoch number and the most recent loss value.

model.train()

print("\nTraining completed!")


**What's happening:**

**Step 1: Move model to GPU (once)**

```python
model = SimpleNN(20, 50, 2).to(device)
# All model weights now on GPU

```

**Step 2: Move each batch to GPU (every iteration)**

```python
X_batch = X_batch.to(device)  # Input data to GPU
y_batch = y_batch.to(device)  # Labels to GPU

```

**Step 3: Everything runs on GPU automatically**

```python
predictions = model(X_batch)  # Computation on GPU
loss = criterion(predictions, y_batch)  # Loss computed on GPU
loss.backward()  # Gradients computed on GPU
optimizer.step()  # Weights updated on GPU

```

**Data flow:**

```
CPU                           GPU
────                          ────
DataLoader
   ↓
X_batch, y_batch  ──.to(device)──→  X_batch, y_batch
                                         ↓
                                    model(X_batch)
                                         ↓
                                    predictions
                                         ↓
                                    loss.backward()
                                         ↓
                                    optimizer.step()
                                         ↓
loss.item()      ←──────────────  loss (scalar)

```

**Why `.item()` works:**

-   Copies single scalar value from GPU to CPU
-   Cheap operation (just one number)
-   Allows printing without moving entire tensors


**GPU vs CPU speed comparison**



In [ ]:
import time

# Large matrix multiplication
size = 5000
A = torch.randn(size, size)
B = torch.randn(size, size)

# CPU timing
start = time.time()
C_cpu = A @ B
cpu_time = time.time() - start

# GPU timing
A_gpu = A.to(device)
B_gpu = B.to(device)
torch.cuda.synchronize()  # Wait for GPU to finish

start = time.time()
C_gpu = A_gpu @ B_gpu
torch.cuda.synchronize()  # Wait for GPU to finish
gpu_time = time.time() - start

print(f"CPU time: {cpu_time:.4f} seconds")
print(f"GPU time: {gpu_time:.4f} seconds")
print(f"Speedup: {cpu_time / gpu_time:.1f}x")


**When GPU helps most:**

-   Large matrix operations (matrix multiply, convolutions)
-   Deep neural networks with many parameters
-   Large batch sizes
-   Many parallel operations

**When GPU may not help:**

-   Small models or data
-   Lots of CPU-GPU data transfer
-   Sequential operations (can't parallelize)


**Best practices for GPU training**

**1. Move model once, move data every batch:**

```python
# ✓ Good
model.to(device)  # Once at start
for X, y in dataloader:
    X, y = X.to(device), y.to(device)  # Every batch

# ✗ Bad (don't move model every iteration)
for X, y in dataloader:
    model.to(device)  # Wasteful!
    X, y = X.to(device), y.to(device)

```

**2. Clear GPU memory when done:**

```python
del model, X_batch, y_batch  # Delete references
torch.cuda.empty_cache()      # Free unused memory

```

**3. Monitor GPU memory:**

```python
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

```

**4. Use device-agnostic code:**

```python
# ✓ Works on CPU or GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# ✗ Hardcoded device (fails if no GPU)
model.to('cuda')  # Error if no GPU!

```

**5. Batch size considerations:**

```python
# GPU can handle larger batches efficiently
batch_size_cpu = 32
batch_size_gpu = 128  # 4x larger

# Rule of thumb: increase batch size until GPU memory ~80% full

```

**Common GPU errors and fixes:**

```
Error: "CUDA out of memory"
Fix: Reduce batch size or model size

Error: "Expected all tensors to be on same device"
Fix: Ensure all tensors moved to same device

Error: "CUDA error: device-side assert triggered"
Fix: Check for invalid indices (e.g., in cross entropy loss)

```


### Memory Management

In [ ]:
if torch.cuda.is_available():
    # Clear GPU cache
    torch.cuda.empty_cache()
    
    # Monitor memory
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
    # Get memory summary
    print("\nMemory Summary:")
    print(torch.cuda.memory_summary(0, abbreviated=True))
else:
    print("No GPU available")



## 8. Practice Exercises

### Exercise 1: Tensor Manipulations

Complete the following tensor operations:

In [ ]:
# YOUR CODE HERE
#
# 1. Create a 3x3 tensor `tensor_a` filled with random values drawn
#    uniformly from the [0, 1) interval.
#
# 2. Create a 3x3 identity matrix `tensor_b` (ones on the diagonal,
#    zeros elsewhere).
#
# 3. Compute the element-wise product of tensor_a and tensor_b and
#    store it in `product_elem`.
#
# 4. Compute the matrix product of tensor_a and tensor_b and store
#    it in `product_mat`.
#
# 5. Compute the transpose of tensor_a and store it in `tensor_a_T`.
#
# 6. Print all four results, each with its shape on the same line.

tensor_a = None
tensor_b = None
product_elem = None
product_mat = None
tensor_a_T = None


### Exercise 2: Build a Custom Network

Create a neural network for MNIST-like classification (28x28 images → 10 classes):

In [ ]:
class MNISTClassifier(nn.Module):
    """A feed-forward MNIST classifier.

    Architecture: a 784-dim input vector (a flattened 28x28 image) flows
    through three fully-connected layers with sizes 784 -> 128 -> 64 -> 10.
    Between the first and second layers, and between the second and third
    layers, apply a ReLU activation followed by a dropout layer with
    p=0.2. The final 10 numbers are class logits, one per MNIST digit
    class (no activation on the output layer).
    """
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        #
        # Define four submodules as attributes on self:
        #   - a Linear that maps 784 features to 128 features
        #   - a Linear that maps 128 features to 64 features
        #   - a Linear that maps 64 features to 10 features
        #   - a Dropout layer with p=0.2 (you can reuse the same one twice
        #     in forward, or create two separate ones -- both work)
        pass

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Pass x through the first Linear, apply ReLU, then dropout.
        # Pass that result through the second Linear, apply ReLU, then
        # dropout again. Finally pass the result through the third Linear
        # and return it (no activation on the output).
        pass


# Sanity check (uncomment after implementing)
# model = MNISTClassifier()
# sample_input = torch.randn(2, 784)   # batch of 2 flattened images
# output = model(sample_input)
# print(f"Output shape: {output.shape}")   # expected: torch.Size([2, 10])


### Exercise 3: Training Loop

Complete the training loop:

In [ ]:
# Setup (using previous data)
model = SimpleNN(20, 50, 2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


def train_one_epoch(model, dataloader, criterion, optimizer):
    """One full pass over the dataloader. Returns the average loss.

    For each batch (batch_X, batch_y) drawn from the dataloader:
      - Zero the optimizer's gradients before the forward pass.
      - Run the forward pass through the model to get outputs.
      - Compute the loss between outputs and batch_y using `criterion`.
      - Run a backward pass to populate gradients.
      - Step the optimizer to update the model weights.
      - Accumulate the batch's scalar loss value into a running total.

    After the loop, return the average loss across the epoch (running
    total divided by the number of batches in the dataloader).
    """
    model.train()
    # YOUR CODE HERE
    pass


# Try it out (uncomment after implementing).
#
# NOTE: This sanity check needs a `dataloader` variable in scope. The
# easiest way is to reuse `train_loader` from the earlier "complete
# training example" cell: pass it as the second argument to
# train_one_epoch and use it as your dataloader. Alternatively, build
# a fresh DataLoader by wrapping a SimpleDataset over some (X, y) data.
#
# avg_loss = train_one_epoch(model, train_loader, criterion, optimizer)
# print(f"Average loss: {avg_loss:.4f}")


### Exercise 4: Embedding Layer

Create a simple embedding-based text classifier:

In [ ]:
class TextClassifier(nn.Module):
    """A bag-of-words style classifier using an embedding layer.

    Architecture: a tensor of token IDs with shape (batch, seq_len) is
    looked up in an embedding table (one row per vocabulary token, one
    column per embedding dimension), producing a (batch, seq_len,
    embedding_dim) tensor. Average across the seq_len dimension to get
    one embedding_dim-sized vector per sample, then pass that through
    two Linear layers with a ReLU between them. The final output has
    shape (batch, num_classes).
    """
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        # YOUR CODE HERE
        #
        # Define three submodules as attributes on self:
        #   - an Embedding layer with vocab_size entries, each of size
        #     embedding_dim
        #   - a Linear that maps embedding_dim features to 64 features
        #   - a Linear that maps 64 features to num_classes features
        pass

    def forward(self, token_ids):
        # token_ids shape: (batch_size, seq_len)
        # YOUR CODE HERE
        #
        # 1. Look up token_ids in the embedding table. The result has
        #    shape (batch, seq_len, embedding_dim).
        # 2. Average across the seq_len dimension to get a tensor of
        #    shape (batch, embedding_dim). This is the "bag of words"
        #    pooled representation.
        # 3. Pass the pooled vector through the first Linear layer and
        #    apply a ReLU activation.
        # 4. Pass the result through the second Linear layer to produce
        #    the class logits.
        # 5. Return the class logits.
        pass


# Sanity check (uncomment after implementing)
# model = TextClassifier(vocab_size=1000, embedding_dim=32, num_classes=5)
# token_ids = torch.randint(0, 1000, (4, 10))   # batch of 4 sequences, 10 tokens each
# logits = model(token_ids)
# print(f"Logits shape: {logits.shape}")        # expected: torch.Size([4, 5])




## Summary: Key Takeaways

### Essential Concepts for LLMs

1. **Tensors**
   - Multi-dimensional arrays (the foundation of everything)
   - Shape manipulation is critical
   - Understanding dimensions: batch × sequence × features

2. **Automatic Differentiation**
   - `.backward()` computes gradients automatically
   - Always zero gradients before backward pass
   - Use `torch.no_grad()` for inference

3. **Neural Network Building Blocks**
   - `nn.Module` is the base class for all models
   - Embedding layers convert token IDs to vectors
   - Linear layers perform transformations
   - LayerNorm and Dropout are used in transformers

4. **Training Loop**
   - Forward → Loss → Backward → Update weights
   - Use DataLoader for batching
   - Track metrics (loss, accuracy)
   - Save checkpoints

5. **GPU Acceleration**
   - Move models and data to GPU with `.to(device)`
   - Essential for training large models
   - Monitor memory usage

### Next Steps for LLM Development

Now you're ready to learn:
- **Attention Mechanisms**: The core of transformers
- **Transformer Architecture**: Multi-head attention, position encodings
- **Tokenization**: Converting text to token IDs
- **Training Strategies**: Learning rate schedules, gradient clipping
- **Model Scaling**: Handling billions of parameters

Continue with the main "LLMs from Scratch" book - you now have the PyTorch foundation!



## Additional Resources

- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)
- [PyTorch Tutorials](https://pytorch.org/tutorials/)
- [Deep Learning with PyTorch](https://pytorch.org/assets/deep-learning/Deep-Learning-with-PyTorch.pdf)
- [Hugging Face Course](https://huggingface.co/course/chapter1/1)



*This notebook was designed to complement "LLMs from Scratch" by Sebastian Raschka*